## Bonus assignment 5 - Decoding Strategies

###  **Objective**

Implement the three most common decoding algorithms used in LLMs using raw PyTorch tensor manipulation:

1.  **Greedy Search:** The deterministic baseline.
2.  **Sampling with Top-K & Nucleus (Top-P):** Controlling randomness and diversity.
3.  **Beam Search:** Exploring multiple future paths to find the most probable sequence.

###  **Assignment Structure**

This assignment will contain:

1.  **Setup Block:** A `SimpleGPT` model class (a tiny Transformer) and a `get_model` function that returns a model with fixed weights (for deterministic value checks).
2.  **Part 1: Greedy Search:** Implementing the basic loop.
3.  **Part 2: Advanced Sampling:** Implementing the logic to filter logits via Top-K and Top-P thresholds.
4.  **Part 3: Beam Search:** Implementing the candidate expansion and pruning logic.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- Configuration ---
VOCAB_SIZE = 100  # Small vocab for easy debugging
D_MODEL = 32
SEQ_LEN = 20

# --- Reproducibility ---
torch.manual_seed(42)

# --- Mock Model ---
# A tiny Transformer-like model that is consistent but untrained.
# We use this so your "Value Checks" will match ours exactly.
class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        # A single Linear layer acting as "attention" + "mlp" mixture
        self.net = nn.Linear(d_model, d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        # idx: (batch, seq_len)
        x = self.embedding(idx)
        x = F.relu(self.net(x))
        logits = self.head(x)
        return logits

def get_model():
    torch.manual_seed(42) # Fixed weights
    model = SimpleGPT(VOCAB_SIZE, D_MODEL)
    model.eval() # Disable dropout/grad
    return model

print("="*50)
print(f"Model Config: Vocab={VOCAB_SIZE}, D_Model={D_MODEL}")
print("="*50)

Model Config: Vocab=100, D_Model=32


-----

### **Part 1: Greedy Search**

**Instructions:** Implement the `greedy_decode` function. At each step, pass the current sequence to the model, get the logits for the *last* position, find the token index with the highest probability (argmax), append it, and repeat.

In [ ]:
print("\n--- Part 1: Greedy Search ---")

def greedy_decode(model, start_ids, max_new_tokens):
    """
    Generates text using greedy search.
    Args:
        model: The language model.
        start_ids: Tensor of shape (1, seq_len) containing prompt.
        max_new_tokens: Number of tokens to generate.
    Returns:
        Tensor of shape (1, seq_len + max_new_tokens)
    """
    # Clone to avoid modifying original
    curr_ids = start_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 1. Get model outputs
            logits = model(curr_ids)

            # 2. Focus only on the last time step
            # Shape: (1, vocab_size)
            next_token_logits = logits[:, -1, :]


            # 3. Select the token with the highest logit (greedy)
            # Shape: (1, 1)
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            # 4. Append to sequence
            # Hint: Use torch.cat
            curr_ids = torch.cat([curr_ids, next_token_id], dim=1)

    return curr_ids

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    model = get_model()
    start_ids = torch.tensor([[10, 20, 30]]) # Dummy prompt
    output = greedy_decode(model, start_ids, max_new_tokens=5)


    assert output.shape == (1, 8)
    print(" (1a) Output shape is correct.")

    # Deterministic check
    expected_last_token = 50
    if output[0, -1].item() == expected_last_token:
        print(" (1b) Generated sequence matches expected deterministic output.")
    else:
        print(f" (1b) Expected last token {expected_last_token}, got {output[0, -1].item()}")

    print(" Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: Greedy Search ---
--- Running Verification for Part 1 ---
 (1a) Output shape is correct.
 (1b) Generated sequence matches expected deterministic output.
 Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Value Check**
Run `greedy_decode` using the `model` from `get_model()` and `start_ids = torch.tensor([[1]])` for `max_new_tokens = 10`. What is the **sum** of the token IDs in the final generated sequence (including the start token)?

In [ ]:
model = get_model()
start_ids = torch.tensor([[1]])
output = greedy_decode(model, start_ids, max_new_tokens=10)
print(f"Q1 Sum of Token IDs: {output.sum().item()}")

Q1 Sum of Token IDs: 425




**Q2: Conceptual Check**
Which of the following is a major downside of Greedy Search?

  * (A) It is computationally expensive compared to Beam Search.
  * (B) It often gets stuck in repetitive loops (e.g., "I went to the the the...").
  * (C) It creates text that is too random and incoherent.
  * (D) It requires calculating the softmax probability distribution.



-----

### **Part 2: Top-K and Top-P Sampling**

**Instructions:** Implement a filtering function.

1.  **Top-K:** Keep only the `k` highest logits; set others to `-float('inf')`.
2.  **Top-P (Nucleus):** Sort logits. Calculate cumulative probability (softmax -\> cumsum). Remove tokens *after* the cumulative probability exceeds `p`.

In [ ]:
print("\n--- Part 2: Top-K and Top-P Sampling ---")

def modify_logits_for_sampling(logits, top_k=None, top_p=None):
    """
    Applies Top-K and/or Top-P filtering to logits.
    Args:
        logits: Shape (batch, vocab_size) - typically the last step
    """
    # Work on a clone
    logits = logits.clone()

    # --- Top-K Filtering ---
    if top_k is not None and top_k > 0:
        # 1. Find the value of the k-th largest logit

        top_k_values, _ = torch.topk(logits, k=top_k)
        min_value = top_k_values[:, -1].unsqueeze(1)


        # 2. Mask logits smaller than min_value

        logits[logits < min_value] = -float('inf')

    # --- Top-P (Nucleus) Filtering ---
    if top_p is not None and top_p > 0.0:
        # 1. Sort logits in descending order
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)

        # 2. Calculate cumulative probabilities
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # 3. Create filter mask
        # Remove tokens with cumulative probability above the threshold
        # (Shift mask right to include the first token that crosses the threshold)
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = 0


        # 4. Scatter sorted mask back to original indices
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)

        # 5. Apply mask
        logits[indices_to_remove] = -float('inf')

    return logits

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
try:
    # Test Top-K
    dummy_logits = torch.tensor([[1.0, 2.0, 8.0, 4.0]]) # Indices: 0, 1, 2, 3
    # If K=2, we keep 8.0 (idx 2) and 4.0 (idx 3). Others becomes -inf
    filtered_k = modify_logits_for_sampling(dummy_logits, top_k=2)
    assert filtered_k[0, 0] == -float('inf')
    assert filtered_k[0, 2] == 8.0
    print(" (2a) Top-K filtering logic is correct.")

    # Test Top-P
    # Probs approx: [0.1, 0.1, 0.8]. Cumsum: [0.8, 0.9, 1.0].
    # If P=0.75, we should only keep the first one (0.8).
    # However, we usually keep the one that crosses the threshold, so maybe just the first.
    dummy_logits_p = torch.tensor([[10.0, 1.0, 1.0]])
    filtered_p = modify_logits_for_sampling(dummy_logits_p, top_p=0.5)
    assert filtered_p[0, 1] == -float('inf') # Should be removed
    assert filtered_p[0, 0] == 10.0 # Should stay
    print(" (2b) Top-P filtering logic is correct.")
    print(" Part 2 seems correct!")

except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: Top-K and Top-P Sampling ---
--- Running Verification for Part 2 ---
 (2a) Top-K filtering logic is correct.
 (2b) Top-P filtering logic is correct.
 Part 2 seems correct!


#### **Questions for Part 2**

**Q3: Value Check (Top-K)**
Given logits `x = torch.tensor([[2.0, 4.0, 3.0, 10.0, 5.0]])` and `top_k=3`. Run `modify_logits_for_sampling`. What is the **sum** of the resulting logits (summing `-inf` will give `-inf`, so only sum the *finite* values)?

In [ ]:
x = torch.tensor([[2.0, 4.0, 3.0, 10.0, 5.0]])
top_k = 3
filtered = modify_logits_for_sampling(x, top_k=top_k)
print(f"Q3 Sum of Filtered Logits: {filtered[filtered != -float('inf')].sum().item()}")

Q3 Sum of Filtered Logits: 19.0



**Q4: Value Check (Top-P)**
Given logits `x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])` and `top_p=0.8`. Run `modify_logits_for_sampling`. How many finite logits remain (count of non-inf values)?

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
top_p = 0.8
filtered = modify_logits_for_sampling(x, top_p=top_p)
print(f"Q4 Count of Finite Logits: {(filtered != -float('inf')).sum().item()}")

Q4 Count of Finite Logits: 2




-----

### **Part 3: Beam Search**

**Instructions:** Implement the core step of Beam Search: **Expand and Prune**.
You will perform one step of generation:

1.  Take `current_beams` (a list of sequences).
2.  Run the model on all beams.
3.  Get the Top-K tokens for *each* beam (where K = num\_beams).
4.  Calculate the *sequence score* (previous score + log\_prob of new token).
5.  Select the global best `num_beams` candidates from all expansions.

In [ ]:
print("\n--- Part 3: Beam Search (Step Logic) ---")

def beam_search_step(model, current_beams, num_beams):
    """
    Performs one step of beam search.
    Args:
        model: The LLM.
        current_beams: List of tuples (score, sequence_tensor).
        num_beams: Beam width (K).
    Returns:
        List of best 'num_beams' tuples (score, sequence_tensor)
    """
    candidates = []

    # 1. Iterate through each active beam
    for score, seq in current_beams:
        with torch.no_grad():
            # Get logits for the next token
            # seq shape: (1, current_len)
            output = model(seq)
            next_token_logits = output[:, -1, :] # (1, vocab)

            # Calculate Log Softmax to get log-probabilities
            next_token_probs = F.log_softmax(next_token_logits, dim=-1)

            # 2. Expand: Get top 'num_beams' tokens for this beam
            top_scores, top_ids = torch.topk(next_token_probs, num_beams)

            # 3. Add candidates to the list
            for i in range(num_beams):
                token_score = top_scores[0, i].item()
                token_id = top_ids[0, i].item()

                # New score = old score + new token log prob
                new_score = score + token_score

                # New sequence
                new_seq = torch.cat([seq, torch.tensor([[token_id]])], dim=1)

                candidates.append((new_score, new_seq))

    # 4. Prune: Sort all candidates by score (descending) and take top 'num_beams'
    # Hint: use sorted(..., key=lambda x: x[0], reverse=True)
    ordered = sorted(candidates, key=lambda x: x[0], reverse=True)
    best_beams = ordered[:num_beams]

    return best_beams

# --- Verification for Part 3 ---
print("--- Running Verification for Part 3 ---")
try:
    model = get_model()
    # Start with one beam: Score 0.0, Sequence [10]
    beams = [(0.0, torch.tensor([[10]]))]

    # Run one step with beam_width=3
    new_beams = beam_search_step(model, beams, num_beams=3)

    assert len(new_beams) == 3
    print(" (3a) Correct number of beams returned.")

    # Check that scores are negative (log probs) and descending
    assert new_beams[0][0] >= new_beams[1][0] >= new_beams[2][0]
    assert new_beams[0][0] < 0
    print(" (3b) Beams are sorted by score correctly.")
    print(" Part 3 seems correct!")
except Exception as e:
    print(f" Part 3 Failed: {e}")


--- Part 3: Beam Search (Step Logic) ---
--- Running Verification for Part 3 ---
 (3a) Correct number of beams returned.
 (3b) Beams are sorted by score correctly.
 Part 3 seems correct!


#### **Questions for Part 3**

**Q5: Conceptual Check**
Why do we use `log_softmax` and addition for scores in Beam Search, rather than standard `softmax` and multiplication?

  * (A) Addition is faster than multiplication on GPUs.
  * (B) Multiplying many small probabilities results in numerical underflow (values becoming 0).
  * (C) Log-softmax normalizes the logits better.
  * (D) It makes the gradients flow backward easier.



**Q6: Value Check**
Run `beam_search_step` using the `get_model()` model.
Start state: `beams = [(0.0, torch.tensor([[50]])) ]` (Start token ID 50).
Parameter: `num_beams = 5`.
What is the **score** of the *best* (0-th index) beam returned?

In [ ]:
# Code for Q6
model_q6 = get_model()
beams_q6 = [(0.0, torch.tensor([[50]]))]
out_q6 = beam_search_step(model_q6, beams_q6, 5)
print(f"Q6 Best Score: {out_q6[0][0]:.4f}")

Q6 Best Score: -3.8822




*(Enter float rounded to 4 decimals)*

**Q7: Value Check**
Using the result from Q6 (`out_q6`), what is the **token ID** of the newly added token in the *best* beam?

In [ ]:
# Code for Q7 (The last token in the best beam's sequence)
last_token = out_q6[0][1][0, -1].item()
print(f"Q7 Last Token ID: {last_token}")

Q7 Last Token ID: 14
